# IOAI — 2025 Stage 2 Source Extraction (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
if not os.path.exists('data/corpus.jsonl'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-2-source-extraction/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 소스 추출 — 밀집 검색 (Source Extraction / Ekstrakcja Źródeł)

폴란드 AI 올림피아드 II · 2025 · 2단계. RAG 의 핵심인 **검색(retrieval)** 문제. 과학적 주장(query)이
주어지면, 문서 코퍼스(corpus)에서 그 주장의 **근거가 된 원본 문서**를 찾아야 한다(SciFact 계열).

**방법**: `Embedder` 를 구현해 쿼리와 문서를 같은 벡터공간에 임베딩 → **코사인 유사도** 상위 k=10 을 반환.
채점은 **nDCG**: 각 쿼리에서 정답 문서가 랭크 i(0-기반)에 있으면 `1/log2(i+2)`, 없으면 0, 쿼리 평균.
`compute_score(nDCG)`: ≤0.2 → 0, 0.2~0.5 선형, ≥0.5 → 100.

이 노트북은 **베이스라인** — 스캐폴드 기본 `Embedder`(모두 1인 임베딩, 미구현 상태)라 nDCG≈0 → 0점.
모범답안(SGPT 밀집 임베더)을 참고해 `encode_queries`/`encode_corpus` 를 구현하라.

**제출**: `submission.csv` — `query_id,doc_ids` (doc_ids = 상위10 text_id 를 랭크순 공백구분).


In [ ]:
# 데이터 준비 (Colab: 자동 다운로드 / DGX: data/ 이미 존재)
import os, urllib.request, zipfile
if not os.path.exists("data/corpus.jsonl"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-2-source-extraction/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")

import json, csv, torch
from transformers import AutoModel, AutoTokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Tokenizer:
    def __init__(self, tokenizer_path, length=150):
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "right"
        self.length = length
    def __call__(self, batch_text):
        return self.tokenizer(batch_text, max_length=self.length, truncation=True,
                              padding=True, return_tensors="pt").to(device)

def load_corpus(file):
    corpus = {}
    for line in open(file, encoding="utf8"):
        o = json.loads(line); corpus[o["text_id"]] = {"text": o.get("text"), "title": o.get("title")}
    return corpus

def load_queries(file):                       # 이 사이트판: query_id + query (정답 미포함)
    queries = {}
    for line in open(file, encoding="utf8"):
        o = json.loads(line); queries[o["query_id"]] = o["query"]
    return queries

corpus = load_corpus("data/corpus.jsonl")
queries = load_queries("data/queries.jsonl")
print(f"corpus {len(corpus)} texts, queries {len(queries)}")


In [ ]:
# 검색(고정 코드) — 코사인 유사도 상위 k
def cos_sim(a, b):
    a = torch.nn.functional.normalize(a, p=2, dim=1)
    b = torch.nn.functional.normalize(b, p=2, dim=1)
    return torch.mm(a, b.transpose(0, 1))

def search_topk_texts(embedder, corpus, queries, top_k=10):
    query_ids = list(queries.keys())
    q_texts = [queries[q] for q in query_ids]
    query_embeddings = embedder.encode_queries(q_texts)
    corpus_ids = list(corpus.keys())
    corpus_texts = [corpus[c] for c in corpus_ids]
    corpus_embeddings = embedder.encode_corpus(corpus_texts)
    cos = cos_sim(query_embeddings, corpus_embeddings)
    cos[torch.isnan(cos)] = -1
    top_val, top_idx = torch.topk(cos, min(top_k, cos.shape[1]), dim=1, largest=True, sorted=True)
    results = {}
    for qi, qid in enumerate(query_ids):
        results[qid] = [corpus_ids[j] for j in top_idx[qi].cpu().tolist()]
    return results

def save_submission(results, path="submission.csv"):
    with open(path, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["query_id", "doc_ids"])
        for qid, ids in results.items():
            w.writerow([qid, " ".join(str(x) for x in ids)])
    print("submission.csv 저장:", len(results), "쿼리")


In [ ]:
class Embedder:
    """베이스라인: 스캐폴드 기본(모두 1인 임베딩). 모든 유사도가 같아 검색이 무의미 → nDCG≈0.
    TODO: encode_queries / encode_corpus 를 실제 임베더로 구현하라(모범답안 참고)."""
    def __init__(self):
        pass
    def encode_queries(self, queries):
        return torch.ones(len(queries), 768).to(device)
    def encode_corpus(self, texts):
        return torch.ones(len(texts), 768).to(device)

embedder = Embedder()
with torch.no_grad():
    results = search_topk_texts(embedder, corpus, queries, top_k=10)
save_submission(results)


### 다음 단계
`Embedder.encode_queries`/`encode_corpus` 를 실제 임베더(SGPT 등)로 구현하라. weighted-mean pooling
+ special brackets 로 쿼리·문서를 768차원에 임베딩하면 nDCG 가 0 → ~0.5 로 오른다. 모범답안 참고.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)